# Phase 1 - Get the data (Electronics 1% sample)

**Project:** Large-Scale Analysis of Amazon Product Reviews using Apache Spark

This notebook:
1. Mounts Google Drive
2. Writes the sampling script to Drive
3. Streams the official Electronics files (7.8 GB) and keeps **all reviews of 1% of users**
4. Copies the sample to Drive
5. Checks the sample

Run the cells **top to bottom**. Cell 4 takes about 15-40 minutes - keep this tab open.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Create project folders on Drive

In [ ]:
import os
PROJECT = '/content/drive/MyDrive/amazon_reviews'
for sub in ['scripts', 'sample']:
    os.makedirs(f'{PROJECT}/{sub}', exist_ok=True)
print(os.listdir(PROJECT))

## 3. Write the sampling script to Drive

Same as `scripts/make_sample.py` in the project folder.

In [ ]:
%%writefile /content/drive/MyDrive/amazon_reviews/scripts/make_sample.py
"""
Build a reduced (sampled) version of the Amazon Reviews 2023 dataset.

Sampling is done by USER: a user is kept if md5(user_id) falls in the chosen
percentage, and then ALL reviews of that user are kept. This keeps each
reviewer's full history, which the fake-review module (reviewer behaviour)
and the ALS recommender (user histories) both need. A plain random sample of
reviews would break those histories.

Metadata is then filtered to only the products that appear in the sample.

The input files are streamed straight from the URL (or read from a local
path), so the full dataset is never stored on disk.

Usage:
    python scripts/make_sample.py                      # 1% of Electronics
    python scripts/make_sample.py --percent 5          # 5% of Electronics
    python scripts/make_sample.py --category Books
    python scripts/make_sample.py --max-lines 100000   # quick test run
"""
import argparse
import gzip
import hashlib
import json
import os
import time
import urllib.request

BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"


def open_source(path_or_url):
    """Return a binary stream of the gzipped file (URL or local path)."""
    if path_or_url.startswith("http"):
        return urllib.request.urlopen(path_or_url, timeout=120)
    return open(path_or_url, "rb")


def keep_user(user_id, percent):
    """Deterministic: the same user is always in or out of the sample."""
    bucket = int(hashlib.md5(user_id.encode("utf-8")).hexdigest()[:8], 16) % 10000
    return bucket < percent * 100


def sample_reviews(src, out_path, percent, max_lines):
    asins = set()
    users = set()
    read = kept = 0
    start = time.time()
    with open_source(src) as raw, gzip.open(raw, "rt", encoding="utf-8") as fin, \
            gzip.open(out_path, "wt", encoding="utf-8") as fout:
        for line in fin:
            read += 1
            review = json.loads(line)
            if keep_user(review["user_id"], percent):
                fout.write(line)
                kept += 1
                asins.add(review["parent_asin"])
                users.add(review["user_id"])
            if read % 1_000_000 == 0:
                print(f"  reviews read: {read:,}  kept: {kept:,}  "
                      f"({time.time() - start:.0f}s)", flush=True)
            if max_lines and read >= max_lines:
                break
    print(f"Reviews done: read {read:,}, kept {kept:,} "
          f"from {len(users):,} users on {len(asins):,} products")
    return asins


def sample_meta(src, out_path, asins, max_lines):
    read = kept = 0
    start = time.time()
    with open_source(src) as raw, gzip.open(raw, "rt", encoding="utf-8") as fin, \
            gzip.open(out_path, "wt", encoding="utf-8") as fout:
        for line in fin:
            read += 1
            if json.loads(line)["parent_asin"] in asins:
                fout.write(line)
                kept += 1
            if read % 500_000 == 0:
                print(f"  products read: {read:,}  kept: {kept:,}  "
                      f"({time.time() - start:.0f}s)", flush=True)
            if max_lines and read >= max_lines:
                break
    print(f"Metadata done: read {read:,}, kept {kept:,}")


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--category", default="Electronics")
    parser.add_argument("--percent", type=float, default=1.0,
                        help="percentage of users to keep (default 1)")
    parser.add_argument("--reviews-src", help="local path or URL (default: official URL)")
    parser.add_argument("--meta-src", help="local path or URL (default: official URL)")
    parser.add_argument("--out-dir", default="data/sample")
    parser.add_argument("--max-lines", type=int, default=0,
                        help="stop after this many input lines (for testing)")
    args = parser.parse_args()

    cat = args.category
    reviews_src = args.reviews_src or f"{BASE_URL}/review_categories/{cat}.jsonl.gz"
    meta_src = args.meta_src or f"{BASE_URL}/meta_categories/meta_{cat}.jsonl.gz"
    tag = f"{args.percent:g}pct"
    os.makedirs(args.out_dir, exist_ok=True)
    reviews_out = os.path.join(args.out_dir, f"{cat}_{tag}.jsonl.gz")
    meta_out = os.path.join(args.out_dir, f"meta_{cat}_{tag}.jsonl.gz")

    print(f"Sampling {args.percent}% of users from {cat}")
    asins = sample_reviews(reviews_src, reviews_out, args.percent, args.max_lines)
    sample_meta(meta_src, meta_out, asins, args.max_lines)
    for path in (reviews_out, meta_out):
        print(f"{path}: {os.path.getsize(path) / 1e6:.1f} MB")


if __name__ == "__main__":
    main()

## 4. Build the 1% sample (15-40 minutes)

Writes to Colab's local disk first (fast), then we copy to Drive.
Progress is printed every 1 million reviews (Electronics has about 43.9 million).

In [ ]:
!python {PROJECT}/scripts/make_sample.py --category Electronics --percent 1 --out-dir /content/sample

## 5. Copy the sample to Drive

In [ ]:
!cp /content/sample/*.gz {PROJECT}/sample/
!ls -lh {PROJECT}/sample/

## 6. Check the sample

pandas is fine here because the sample is small. From Phase 2 onward we use PySpark.

In [ ]:
import pandas as pd
reviews = pd.read_json(f'{PROJECT}/sample/Electronics_1pct.jsonl.gz', lines=True)
meta = pd.read_json(f'{PROJECT}/sample/meta_Electronics_1pct.jsonl.gz', lines=True)

print('Reviews  :', len(reviews))
print('Users    :', reviews['user_id'].nunique())
print('Products :', reviews['parent_asin'].nunique())
print('Meta rows:', len(meta))
print('Years    :', pd.to_datetime(reviews['timestamp'], unit='ms').dt.year.min(),
      'to', pd.to_datetime(reviews['timestamp'], unit='ms').dt.year.max())
print('Verified :', round(reviews['verified_purchase'].mean() * 100, 1), '%')
print('\nRating distribution (%):')
print((reviews['rating'].value_counts(normalize=True).sort_index() * 100).round(1))

In [ ]:
reviews[['rating', 'title', 'text', 'verified_purchase', 'helpful_vote']].head()

In [ ]:
meta[['parent_asin', 'title', 'main_category', 'average_rating', 'rating_number', 'price', 'store']].head()

## Done

Send the output of section 6 back, then move on to **Phase 2 (cleaning with PySpark)**.

- Session dropped? Re-run sections 1, 2 and 4 - the sample is deterministic, nothing is lost.
- Bigger sample later: change `--percent 1` to `--percent 5` (files get a `5pct` name).
- Share the `amazon_reviews` Drive folder with the team.